# Register Model

## Notebook Overview

- Install and Import Libraries
- Start Execution
- Configure Settings
- Log the Model to MLFlow
- Fetch the Latest Model Version from MLflow
- Load the Model and Run Inference

# Install and Import Libraries

In [1]:
%%time

%pip install -r ../requirements.txt --quiet 

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/deep_ep-1.0.0+a84a248-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.2.dev0-py3.12.egg 

In [2]:
# -----------------------------
# Standard library imports
# -----------------------------
import json                 # JSON parsing and serialization
import os                   # Operating system utilities (paths, env vars, etc.)
import re                   # Regular expression utilities
import sys                  # Python runtime environment manipulation
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths
import time

# -----------------------------
# Data manipulation libraries
# -----------------------------
import numpy as np           # Numerical computations and arrays
import pandas as pd          # Data manipulation and analysis
from tabulate import tabulate  # Pretty-print tabular data

# -----------------------------
# Deep learning frameworks
# -----------------------------
import torch                 # PyTorch deep learning framework

# -----------------------------
# Experiment tracking (MLflow)
# -----------------------------
import mlflow                # ML lifecycle management and experiment tracking
import mlflow.pyfunc         # MLflow pyfunc interface for custom models
from mlflow import MlflowClient  # Client interface for MLflow operations
from mlflow.models.signature import ModelSignature  # Model signature definitions
from mlflow.tracking import MlflowClient           # Experiment tracking client
from mlflow.types.schema import (                  # Schema utilities for inputs/outputs
    ColSpec,
    ParamSchema,
    ParamSpec,
    Schema,
    TensorSpec,
)

# -----------------------------
# Local imports
# -----------------------------
# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import logger, download_from_s3_uri  # Project-specific utilities
from src.bert_recommendation_service import BERTTourismModel  # Custom BERT-based recommendation model

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Start Execution

In [3]:
start_time = time.time()
logger.info("Notebook execution started.")

# Configure Settings

In [4]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [5]:
CORPUS_URI = f"s3://149536453923-hpaistudio-public-assets/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/corpus.csv"
EMBEDDINGS_URI = f"s3://149536453923-hpaistudio-public-assets/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/embeddings.csv"

CORPUS_PATH = "../data/raw/corpus.csv"
TOKENIZER_DIR = "../artifacts/tokenizer"
BERT_MODEL_NAME = "bert-large-uncased"
BERT_MODEL_DATAFABRIC_PATH = "/home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo"
EMBEDDINGS_OUTPUT_PATH = "../data/processed/"
BERT_MODEL_ONLINE_PATH = "/root/.cache/torch/NeMo/NeMo_1.22.0/bertlargeuncased/ca4ebba9f05a8ffb79845249ca046983/bertlargeuncased.nemo"
DEMO_PATH = "../demo"
EMBEDDINGS_PATH = "../data/processed/embeddings.csv"
CONFIG_PATH = "../configs/config.yaml"

# Define required constants for MLflow registration
EXPERIMENT_NAME = "BERT_Tourism_Experiment"
RUN_NAME = "BERT_Tourism_Run"
MODEL_NAME = "BERT_Tourism_Model"

In [6]:
%%time

saved_data = download_from_s3_uri(s3_uri=CORPUS_URI, local_path="../data/raw")
logger.info(f"Saved to: {saved_data}")

saved_data = download_from_s3_uri(s3_uri=EMBEDDINGS_URI, local_path="../data/processed")
logger.info(f"Saved to: {saved_data}")

CPU times: user 3.48 s, sys: 687 ms, total: 4.16 s
Wall time: 3.26 s


# Register and Log the Model to MLFlow

In [7]:
%%time

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))

# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Log the BERT similarity model to MLflow
    BERTTourismModel.log_model(
        model_name=MODEL_NAME,
        corpus_path=CORPUS_PATH,
        embeddings_path=EMBEDDINGS_PATH,
        tokenizer_dir=TOKENIZER_DIR,
        bert_model_online_path=BERT_MODEL_ONLINE_PATH,
        bert_model_datafabric_path=BERT_MODEL_DATAFABRIC_PATH,
        demo_path=DEMO_PATH,
        config_path=CONFIG_PATH,
    )
    # Register the logged model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

2025/12/17 17:08:10 INFO mlflow.tracking.fluent: Experiment with name 'BERT_Tourism_Experiment' does not exist. Creating a new experiment.


2025-12-17 17:08:12.987445664 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


[NeMo W 2025-12-17 17:09:30 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    data_file: /home/yzhang/data/nlp/bert/47316/hdf5/lower_case_1_seq_len_512_max_pred_80_masked_lm_prob_0.15_random_seed_12345_dupe_factor_5_shard_1472_test_split_10/books_wiki_en_corpus/training/
    max_predictions_per_seq: 80
    batch_size: 16
    shuffle: true
    num_samples: -1
    num_workers: 2
    drop_last: false
    pin_memory: false
    
[NeMo W 2025-12-17 17:09:30 nemo_logging:405] bert-large-uncased is not in get_pretrained_lm_models_list(include_external=False), will be using AutoModel from HuggingFace.
[NeMo W 2025-12-17 17:09:32 nemo_logging:405] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2025-12-17 17:09:32 nemo_logging:393] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 4.375e-05
        maximize: False
        weight_decay: 0.01
    )


[NeMo W 2025-12-17 17:09:32 nemo_logging:405] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


[NeMo I 2025-12-17 17:09:34 nemo_logging:393] Model BERTLMModel was successfully restored from /home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo.


Successfully registered model 'BERT_Tourism_Model'.
Created version '1' of model 'BERT_Tourism_Model'.


CPU times: user 45.2 s, sys: 20.5 s, total: 1min 5s
Wall time: 3min 46s


# Fetch the Latest Model Version from MLflow

In [8]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the "BERT_Tourism_Model" model (not yet in a specific stage)
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
latest_version = versions[0].version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

# Print the latest model version and its signature
print(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
print(f"Signature: {model_info.signature}")

Latest registered version of 'BERT_Tourism_Model': 1
Signature: inputs: 
  ['query': string (required)]
outputs: 
  ['List of Pledges and Similarities': Tensor('object', (-1,))]
params: 
  ['show_score': boolean (default: False)]



# Load the Model and Run Inference

In [9]:
%%time

# Load the trained BERT similarity model from MLflow
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
print(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

# Define a sample query for testing
query = "Give me a resort budget vacation suggestion"

# Use the model to predict similar results based on the query
result = model.predict({"query": [query]})

[NeMo W 2025-12-17 17:13:09 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    data_file: /home/yzhang/data/nlp/bert/47316/hdf5/lower_case_1_seq_len_512_max_pred_80_masked_lm_prob_0.15_random_seed_12345_dupe_factor_5_shard_1472_test_split_10/books_wiki_en_corpus/training/
    max_predictions_per_seq: 80
    batch_size: 16
    shuffle: true
    num_samples: -1
    num_workers: 2
    drop_last: false
    pin_memory: false
    
[NeMo W 2025-12-17 17:13:09 nemo_logging:405] bert-large-uncased is not in get_pretrained_lm_models_list(include_external=False), will be using AutoModel from HuggingFace.
[NeMo W 2025-12-17 17:13:10 nemo_logging:405] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2025-12-17 17:13:10 nemo_logging:393] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 4.375e-05
        maximize: False
        weight_decay: 0.01
    )


[NeMo W 2025-12-17 17:13:10 nemo_logging:405] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


[NeMo I 2025-12-17 17:13:11 nemo_logging:393] Model BERTLMModel was successfully restored from /phoenix/mlflow/832978540656372044/ba09756b75d2401e83e3bd4d689275db/artifacts/BERT_Tourism_Model/artifacts/bertlargeuncased.nemo.
Successfully loaded model 'BERT_Tourism_Model' version 1 for inference.
CPU times: user 30 s, sys: 4.58 s, total: 34.6 s
Wall time: 1min 15s


In [10]:
# Convert the result into a pandas DataFrame
df = pd.DataFrame(result)

# Drop unnecessary columns if needed
df = df.drop(columns=["Unnamed: 0", "Topic"], errors="ignore")

# Rename columns for better readability
df.rename(columns={"Pledge": "Recommended Option", "Similarity": "Relevance Score"}, inplace=True)

# Display the DataFrame in a tabular format
print(tabulate(df, headers="keys", tablefmt="fancy_grid"))

╒════╤═════════════════════════════════════════════════════════════════════════════════════════════════════╤═══════════════════╕
│    │ Recommended Option                                                                                  │   Relevance Score │
╞════╪═════════════════════════════════════════════════════════════════════════════════════════════════════╪═══════════════════╡
│  0 │ For a budget-friendly vacation, consider a resort with vacation options and cruise activities.      │          0.869199 │
├────┼─────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────┤
│  1 │ For a budget-friendly vacation, consider a getaway with beach options and vacation activities.      │          0.863919 │
├────┼─────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────┤
│  2 │ For a budget-friendly vacation, consider a getaway with hotel options and vacation activit

In [11]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).